In [3]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from dataclasses import dataclass
from typing import List, Optional
import random

In [1]:
min([10, 12, 2])

2

In [ ]:
@dataclass
class Charger:
    id: int
    EVs: List[EV] = []     # EVs assigned to this charging pile
    next_event: float      # Next change in charging

    def __post_init__(self):
        self.next_event: float = min(EVs)

@dataclass
class EV:
    id: int
    c_b: float       # Battery capacity (kWh)
    c_rate: float = 1 # C-rate: p_m = C_b * C_rate
    s_i: float    # Initial SoC (0.0 - 1.0)
    s_th: float  # Taper SoC
    s_f: float   # Target SoC (0.0 - 1.0)
    t_now: float = 0 # Time since the vehicle has plugged in to the charger
    delta_t: float = 0 # Time between to events of interest 
    arrival_time: float

    def __post_init__(self):
        self.p_m = self.c_b * self.c_rate    # Peak charging acceptance rate (kW)
        self.s_now = self.s_i               # Current SoC
        self.s_next = self.s_f              # Next SoC of interest
        self.t_th = self.c_b / self.p_m * (self.s_th - self.s_i)    # Time corresponding to s_thresh
        self.s_i_linear = self.s_i  # SoC upon starting the charge at the linear section of P-s graph
        self.p_d_linear = self.p_m  # Initialiaze linear power demand attribute

    # --- State (changes over time) ---
    def update_power_demand(self, power_supply: float, other_ev: EV = None):
        """
        Updated p_d as a function of presence of other EV
        & available power supply based on current conditions
        """
        total_demand = self.p_m + (other_ev.p_m if other_ev else 0)
        self.p_d_linear = self.p_m * power_supply / max(total_demand, power_supply)

    @property
    def s_taper(self):
        """
        Computes taper SoC based on the power demand
        """
        return 1 - self.p_d_linear / self.p_m * (1 - self.s_th)


    def update_delta_t(self, other_ev: 'EV'):
        linear_term = min(self.s_taper, self.s_next) - min(self.s_taper, self.s_now)
        log_term = - (1 - self.s_taper) * np.log((1 - max(self.s_taper, self.s_next)) / (1 - max(self.s_taper, self.s_now)))
        self.delta_t = self.c_b / self.p_d_linear * (linear_term + log_term)

    @property
    def t_taper(self):
        """
        Computes taper time based on the
        """
        return self.c_b / self.p_d_linear * (self.s_taper - self.s_i_linear)
    
    @property
    def p_soc(self):
        return self.p_d_linear * (1 - max(self.s_now, self.s_taper)) / (1 - self.s_taper)
    
    @property
    def p_time(self):
        return self.p_d_linear * np.exp(-self.p_d_linear / (self.c_b * (1 - self.s_taper)) * (max(self.t_now, self.t_taper) - self.t_taper))
    

class ChargingStationEnv(gym.Env):
    def __init__(self, n=4, m=2, c_p=5, p=50, c_q=10, lam=5.0, max_time=1440):
        super().__init__()
        
        # System Parameters
        self.n = n          # Number of piles
        self.m = m          # Nozzles per pile
        self.c_p = c_p      # Max power bricks per pile
        self.p = p          # kW per brick
        self.max_pile_power = self.c_p * self.p # Total kW available per pile
        self.c_q = c_q      # Queue capacity
        self.lam = lam      # Arrival rate (mean minutes); Number of arrivals in any interval of length t is Poisson(t / lambda)
        # and each inter-arrival time is Exp(1 / lambda)
        self.max_time = max_time # 24 hours in minutes
        
        # EV Distribution Parameters
        self.c_b_options = [50.0, 100.0, 150.0]

        # Action Space: 0 to n-1 (assign to pile), n (Wait / Advance Time)
        self.action_space = spaces.Discrete(self.n + 1) 

        # State tracking
        self.current_time = 0
        self.queue: List[EV] = []
        self.piles: List[List[EV]] = [[] for _ in range(self.n)]
        self.arrival_schedule = {}
        self.finished_evs: List[EV] = [] # For Metric Collection
        self.dropped_cars = 0
        # Track history of L (Cars in system) for Little's Law
        self.history_L = []
        # Utilization and Energy Tracking of Piles and Nozzles
        self.pile_active_minutes = np.zeros(self.n)
        self.nozzle_active_minutes = np.zeros((self.n, self.m))
        self.pile_energy_sold = np.zeros(self.n)
        self.nozzle_energy_sold = np.zeros((self.n, self.m))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            np.random.seed(seed)
            
        self.current_time = 0
        self.queue = []
        self.piles = [[] for _ in range(self.n)]
        self.finished_evs = []
        self.dropped_cars = 0
        self.history_L = []

        self.pile_active_minutes = np.zeros(self.n)
        self.nozzle_active_minutes = np.zeros((self.n, self.m))
        self.pile_energy_sold = np.zeros(self.n)
        self.nozzle_energy_sold = np.zeros((self.n, self.m))
        
        # Pre-generate the entire day's arrivals
        self._pre_generate_arrivals()
        
        # Load minute 0 arrivals
        self._load_arrivals_for_current_time()
        
        return self._get_obs(), {}

    def _pre_generate_arrivals(self):
        """Generates all EV arrivals for the simulation duration."""
        self.arrival_schedule = {t: [] for t in range(self.max_time)}
        ev_counter = 0
        
        for t in range(self.max_time):
            # Poisson distribution for number of arrivals this minute
            num_arrivals = np.random.poisson(1.0 / self.lam)
            for _ in range(num_arrivals):
                ev_counter += 1
                c_b = np.random.choice(self.c_b_options)
                s_init = np.random.uniform(0.10, 0.30)
                s_final = np.random.uniform(0.70, 0.90)
                
                energy_needed = c_b * (s_final - s_init)
                
                new_ev = EV(ev_counter, c_b, s_init, s_final, t, energy_needed=energy_needed)
                self.arrival_schedule[t].append(new_ev)

    def _load_arrivals_for_current_time(self):
        """Moves pre-generated EVs into the queue."""
        if self.current_time in self.arrival_schedule:
            for ev in self.arrival_schedule[self.current_time]:
                if len(self.queue) < self.c_q:
                    self.queue.append(ev)
                else:
                    self.dropped_cars += 1 # Metric: Queue full

    def _get_obs(self):
        return {
            "time": self.current_time,
            "queue_len": len(self.queue),
            "pile_occupancy": [len(pile) for pile in self.piles],
            "head_ev": self.queue[0] if self.queue else None
        }

    def step(self, action):
        reward = 0
        
        # --- CASE 1: Instant Assignment (No time passes) ---
        if action < self.n:
            if len(self.queue) > 0:
                target_pile = self.piles[action]
                if len(target_pile) < self.m:
                    ev = self.queue.pop(0)
                    ev.service_start_time = self.current_time # METRIC: Log wait end
                    target_pile.append(ev)
                    reward += 1
                else:
                    reward -= 1 # Penalty for illegal move (pile full)
            else:
                reward -= 1 # Penalty for trying to assign from empty queue
                
            # Return immediately without advancing time
            return self._get_obs(), reward, False, False, {}

        # --- CASE 2: Wait / Advance Time (Action == n) ---
        # Log 'L' (Cars in system) BEFORE advancing time
        # L = Cars in Queue + Cars currently charging
        current_L = len(self.queue) + sum(len(p) for p in self.piles)
        self.history_L.append(current_L)

        self.current_time += 1
        
        # 1. Update Charging & Check Departures
        for pile_idx, pile in enumerate(self.piles):
            pile_occupancy = len(pile)
            if pile_occupancy > 0:

                # TRACKING: Pile is active this minute
                self.pile_active_minutes[pile_idx] += 1
                # Dynamic Power Sharing: Total power divided by active EVs on pile
                power_per_ev = self.max_pile_power / pile_occupancy 
                energy_per_min = power_per_ev / 60.0 # kWh delivered per minute
                
                # Iterate over EVs and their 'nozzle' index
                for nozzle_idx, ev in enumerate(pile[:]):

                    # TRACKING: Nozzle is active and selling energy
                    self.nozzle_active_minutes[pile_idx][nozzle_idx] += 1
                    self.nozzle_energy_sold[pile_idx][nozzle_idx] += energy_per_min
                    self.pile_energy_sold[pile_idx] += energy_per_min # Aggregate for pile

                    ev.energy_received += energy_per_min
                    
                    if ev.energy_received >= ev.energy_needed:
                        pile.remove(ev)
                        ev.departure_time = self.current_time # METRIC: Log departure
                        self.finished_evs.append(ev)
                        reward += 10 # Reward for finishing
        
        # 2. Load new arrivals for this new minute
        self._load_arrivals_for_current_time()

        terminated = self.current_time >= self.max_time 
        
        return self._get_obs(), reward, terminated, False, {}

def fifo_heuristic_agent(obs, n, m):
    """
    Revised logic: If there's a car in queue and space in a pile, ASSIGN.
    Otherwise, ADVANCE TIME (action = n).
    """
    queue_len = obs["queue_len"]
    pile_occupancy = obs["pile_occupancy"]
    
    # If queue is empty, we must advance time.
    if queue_len == 0:
        return n 
        
    available_piles = [i for i, occ in enumerate(pile_occupancy) if occ < m]
    
    # If queue has cars but all piles are full, we must advance time.
    if not available_piles:
        return n 
        
    # Standard FIFO logic: pick the least occupied pile
    min_occ = min([pile_occupancy[i] for i in available_piles])
    best_piles = [i for i in available_piles if pile_occupancy[i] == min_occ]
    
    return random.choice(best_piles)   

In [ ]:

# --- Data Structures with Metric Tracking ---
@dataclass
class EV:
    id: int
    c_b: float       # Battery capacity (kWh)
    s_init: float    # Initial SoC (0.0 - 1.0)
    s_final: float   # Target SoC (0.0 - 1.0)
    arrival_time: int
    energy_needed: float = 0.0     # kWh
    energy_received: float = 0.0   # kWh
    service_start_time: Optional[int] = None
    departure_time: Optional[int] = None

class ChargingStationEnv(gym.Env):
    def __init__(self, n=4, m=2, c_p=5, p=50, c_q=10, lam=5.0, max_time=1440):
        super().__init__()
        
        # System Parameters
        self.n = n          # Number of piles
        self.m = m          # Nozzles per pile
        self.c_p = c_p      # Max power bricks per pile
        self.p = p          # kW per brick
        self.max_pile_power = self.c_p * self.p # Total kW available per pile
        self.c_q = c_q      # Queue capacity
        self.lam = lam      # Arrival rate (mean minutes); Number of arrivals in any interval of length t is Poisson(t / lambda)
        # and each inter-arrival time is Exp(1 / lambda)
        self.max_time = max_time # 24 hours in minutes
        
        # EV Distribution Parameters
        self.c_b_options = [50.0, 100.0, 150.0]

        # Action Space: 0 to n-1 (assign to pile), n (Wait / Advance Time)
        self.action_space = spaces.Discrete(self.n + 1) 

        # State tracking
        self.current_time = 0
        self.next_event = 0
        self.queue: List[EV] = []
        self.piles: List[List[EV]] = [[] for _ in range(self.n)]
        self.arrival_schedule = {}
        self.finished_evs: List[EV] = [] # For Metric Collection
        self.dropped_cars = 0
        # Track history of L (Cars in system) for Little's Law
        self.history_L = []
        # Utilization and Energy Tracking of Piles and Nozzles
        self.pile_active_minutes = np.zeros(self.n)
        self.nozzle_active_minutes = np.zeros((self.n, self.m))
        self.pile_energy_sold = np.zeros(self.n)
        self.nozzle_energy_sold = np.zeros((self.n, self.m))
        # RNG (initialized in reset)
        self.rng_arrival: np.random.Generator | None = None
        self.rng_choice: np.random.Generator | None = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng_arrival = np.random.default_rng(seed)
            self.rng_choice = np.random.default_rng(seed+240)
            
        self.current_time = 0
        self.queue = []
        self.piles = [[] for _ in range(self.n)]
        self.finished_evs = []
        self.dropped_cars = 0
        self.history_L = []

        self.pile_active_minutes = np.zeros(self.n)
        self.nozzle_active_minutes = np.zeros((self.n, self.m))
        self.pile_energy_sold = np.zeros(self.n)
        self.nozzle_energy_sold = np.zeros((self.n, self.m))
        
        # Pre-generate the entire day's arrivals
        self._pre_generate_arrivals()
        
        # Load minute 0 arrivals
        self._load_arrivals_for_current_time()
        
        return self._get_obs(), {}

    def _pre_generate_arrivals(self):
        """Generates all EV arrivals for the simulation duration."""
        # First sample inter-arrival times from the exp distribution
        arrivals = self.rng_arrival.exponential(self.lam, self.max_time / self.lam * 1.5).cumsum()
        arrivals = arrivals[arrivals <= self.max_time].tolist()
        ev_counter = 0
        
        for t in arrivals:
            ev_counter += 1
            c_b = self.rng_arrival.choice(self.c_b_options)
            s_init = self.rng_arrival.uniform(0.10, 0.30)
            s_final = self.rng_arrival.uniform(0.70, 0.90)
            
            energy_needed = c_b * (s_final - s_init)
            
            new_ev = EV(ev_counter, c_b, s_init, s_final, t, energy_needed=energy_needed)
            self.arrival_schedule[t] = new_ev

    def _load_arrivals_for_current_time(self):
        """Moves pre-generated EVs into the queue."""
        if self.current_time in self.arrival_schedule:
            for ev in self.arrival_schedule[self.current_time]:
                if len(self.queue) < self.c_q:
                    self.queue.append(ev)
                else:
                    self.dropped_cars += 1 # Metric: Queue full

    def _get_obs(self):
        return {
            "time": self.current_time,
            "queue_len": len(self.queue),
            "pile_occupancy": [len(pile) for pile in self.piles],
            "head_ev": self.queue[0] if self.queue else None
        }

    def step(self, action):
        reward = 0
        
        # --- CASE 1: Instant Assignment (No time passes) ---
        if action < self.n:
            if len(self.queue) > 0:
                target_pile = self.piles[action]
                if len(target_pile) < self.m:
                    ev = self.queue.pop(0)
                    ev.service_start_time = self.current_time # METRIC: Log wait end
                    target_pile.append(ev)
                    reward += 1
                else:
                    reward -= 1 # Penalty for illegal move (pile full)
            else:
                reward -= 1 # Penalty for trying to assign from empty queue
                
            # Return immediately without advancing time
            return self._get_obs(), reward, False, False, {}

        # --- CASE 2: Wait / Advance Time (Action == n) ---
        # Log 'L' (Cars in system) BEFORE advancing time
        # L = Cars in Queue + Cars currently charging
        current_L = len(self.queue) + sum(len(p) for p in self.piles)
        self.history_L.append(current_L)

        self.current_time += 1
        
        # 1. Update Charging & Check Departures
        for pile_idx, pile in enumerate(self.piles):
            pile_occupancy = len(pile)
            if pile_occupancy > 0:

                # TRACKING: Pile is active this minute
                self.pile_active_minutes[pile_idx] += 1
                # Dynamic Power Sharing: Total power divided by active EVs on pile
                power_per_ev = self.max_pile_power / pile_occupancy 
                energy_per_min = power_per_ev / 60.0 # kWh delivered per minute
                
                # Iterate over EVs and their 'nozzle' index
                for nozzle_idx, ev in enumerate(pile[:]):

                    # TRACKING: Nozzle is active and selling energy
                    self.nozzle_active_minutes[pile_idx][nozzle_idx] += 1
                    self.nozzle_energy_sold[pile_idx][nozzle_idx] += energy_per_min
                    self.pile_energy_sold[pile_idx] += energy_per_min # Aggregate for pile

                    ev.energy_received += energy_per_min
                    
                    if ev.energy_received >= ev.energy_needed:
                        pile.remove(ev)
                        ev.departure_time = self.current_time # METRIC: Log departure
                        self.finished_evs.append(ev)
                        reward += 10 # Reward for finishing
        
        # 2. Load new arrivals for this new minute
        self._load_arrivals_for_current_time()

        terminated = self.current_time >= self.max_time 
        
        return self._get_obs(), reward, terminated, False, {}

def fifo_heuristic_agent(obs, n, m, rng):
    """
    Revised logic: If there's a car in queue and space in a pile, ASSIGN.
    Otherwise, ADVANCE TIME (action = n).
    """
    queue_len = obs["queue_len"]
    pile_occupancy = obs["pile_occupancy"]
    
    # If queue is empty, we must advance time.
    if queue_len == 0:
        return n 
        
    available_piles = [i for i, occ in enumerate(pile_occupancy) if occ < m]
    
    # If queue has cars but all piles are full, we must advance time.
    if not available_piles:
        return n 
        
    # Standard FIFO logic: pick the least occupied pile
    min_occ = min([pile_occupancy[i] for i in available_piles])
    best_piles = [i for i in available_piles if pile_occupancy[i] == min_occ]
    
    return rng.choice(best_piles)   

In [12]:
def report_metrics(env):
    """Calculates and prints standard queueing metrics."""
    served_evs = env.finished_evs
    num_served = len(served_evs)
    
    if num_served == 0:
        print("No EVs were served.")
        return

    wait_times = [ev.service_start_time - ev.arrival_time for ev in served_evs]
    service_times = [ev.departure_time - ev.service_start_time for ev in served_evs]
    total_time_in_sys = [ev.departure_time - ev.arrival_time for ev in served_evs]

    print(f"--- 24-Hour Simulation Results ---")
    print(f"Mean Inter-arrival Rate: {env.lam} min")
    print(f"Total Arrivals Simulated: {sum(len(q) for q in env.arrival_schedule.values())}")
    print(f"Total Cars Served:        {num_served}")
    print(f"Cars Dropped (Queue Full):{env.dropped_cars}")
    print(f"Cars Remaining in System: {sum(len(q) for q in env.piles) + len(env.queue)}")
    print("------------------------------------")
    print(f"Avg Wait Time in Queue ($W_q$): {np.mean(wait_times):.2f} mins")
    print(f"Avg Service/Charge Time:      {np.mean(service_times):.2f} mins")
    print(f"Avg Total Time in Sys ($W$):    {np.mean(total_time_in_sys):.2f} mins")

    # Calculate Utilization Rates (%)
    total_minutes = env.max_time
    pile_utilization = (env.pile_active_minutes / total_minutes) * 100
    nozzle_utilization = (env.nozzle_active_minutes / total_minutes) * 100

    print("\n--- PILE & NOZZLE METRICS ---")
    print(f"{'Pile':<6} | {'Pile Util (%)':<15} | {'Pile Energy (kWh)':<18} | {'Nozzle 0 (Util/kWh)':<20} | {'Nozzle 1 (Util/kWh)':<20}")
    print("-" * 88)
    
    for i in range(env.n):
        # Format Nozzle 0 stats
        n0_util = nozzle_utilization[i][0]
        n0_energy = env.nozzle_energy_sold[i][0]
        n0_str = f"{n0_util:.1f}% / {n0_energy:.1f}"
        
        # Format Nozzle 1 stats (Assuming m=2 for display)
        n1_util = nozzle_utilization[i][1] if env.m > 1 else 0.0
        n1_energy = env.nozzle_energy_sold[i][1] if env.m > 1 else 0.0
        n1_str = f"{n1_util:.1f}% / {n1_energy:.1f}"

        print(f"P-{i:<4} | {pile_utilization[i]:<15.1f} | {env.pile_energy_sold[i]:<18.1f} | {n0_str:<20} | {n1_str:<20}")
    
    print("-" * 88)
    total_energy = np.sum(env.pile_energy_sold)
    print(f"Total Energy Sold: {total_energy:.2f} kWh")
    
def perform_sanity_checks(env):
    """
    Checks the validity of the simulation using Little's Law 
    and Energy Conservation principles.
    """
    print("\n--- ANALYTICAL SANITY CHECKS ---")
    
    # 1. LITTLE'S LAW CHECK: L = lambda_eff * W
    
    if not env.history_L:
        print("Error: No L history tracked. Cannot perform Little's Law check.")
        return

    # Calculate Avg L (Time-average number of cars in system)
    avg_L_simulated = np.mean(env.history_L)
    
    # Calculate effective lambda (Throughput)
    total_time_mins = env.max_time
    total_served = len(env.finished_evs)
    lambda_eff = total_served / total_time_mins
    
    # Average time in system (W)
    if total_served > 0:
        total_time_in_sys = [ev.departure_time - ev.arrival_time for ev in env.finished_evs]
        avg_W_simulated = np.mean(total_time_in_sys)
    else:
        avg_W_simulated = 0
    
    L_theoretical = lambda_eff * avg_W_simulated
    
    print(f"[Little's Law Check]")
    print(f"  Simulated Avg L (from history):   {avg_L_simulated:.4f} cars")
    print(f"  Theoretical L (Throughput * W):   {L_theoretical:.4f} cars")
    
    # The discrepancy should be small (< 0.5 cars usually, depending on start/end transients)
    diff = abs(avg_L_simulated - L_theoretical)
    print(f"  Difference:                       {diff:.4f}")
    
    if diff < 0.2:
        print("  >> PASS: Little's Law holds.")
    else:
        print("  >> WARNING: Discrepancy detected (normal if system hasn't reached steady state).")

    # 2. ENERGY CONSERVATION CHECK
    # Energy Needed by Served Cars vs Energy Sold by Piles
    total_energy_needed = sum([ev.c_b * (ev.s_final - ev.s_init) for ev in env.finished_evs])
    total_energy_sold = np.sum(env.pile_energy_sold)
    
    print(f"\n[Energy Conservation Check]")
    print(f"  Total Energy Demand (Served EVs): {total_energy_needed:.2f} kWh")
    print(f"  Total Energy Sold (Piles):        {total_energy_sold:.2f} kWh")
    
    # Allow small floating point error
    if abs(total_energy_needed - total_energy_sold) < 1.0: 
         print("  >> PASS: Energy In = Energy Out.")
    else:
         print("  >> FAIL: Energy conservation violated.")

In [62]:
# --- RUN THE SIMULATION ---
# n: Number of piles, m: Nozzles per pile, c_p: Max power bricks per pile, p: kW per brick, c_q: Queue capacity
env = ChargingStationEnv(n=4, m=2, c_p=5, p=50, c_q=100, lam=3.0, max_time=14400)
obs, _ = env.reset(seed=41) # Seeding for reproducibility

terminated = False
while not terminated:
    action = fifo_heuristic_agent(obs, env.n, env.m, env.rng_choice)
    obs, reward, terminated, _, info = env.step(action)

In [63]:
report_metrics(env)

--- 24-Hour Simulation Results ---
Mean Inter-arrival Rate: 3.0 min
Total Arrivals Simulated: 4725
Total Cars Served:        3934
Cars Dropped (Queue Full):692
Cars Remaining in System: 99
------------------------------------
Avg Wait Time in Queue ($W_q$): 335.02 mins
Avg Service/Charge Time:      29.24 mins
Avg Total Time in Sys ($W$):    364.26 mins

--- PILE & NOZZLE METRICS ---
Pile   | Pile Util (%)   | Pile Energy (kWh)  | Nozzle 0 (Util/kWh)  | Nozzle 1 (Util/kWh) 
----------------------------------------------------------------------------------------
P-0    | 100.0           | 59983.3            | 100.0% / 30002.1     | 99.9% / 29981.2     
P-1    | 100.0           | 60000.0            | 100.0% / 30029.2     | 99.9% / 29970.8     
P-2    | 100.0           | 60000.0            | 100.0% / 30027.1     | 99.9% / 29972.9     
P-3    | 100.0           | 59987.5            | 100.0% / 29995.8     | 100.0% / 29991.7    
-----------------------------------------------------------------

In [32]:
perform_sanity_checks(env)


--- ANALYTICAL SANITY CHECKS ---
[Little's Law Check]
  Simulated Avg L (from history):   97.8787 cars
  Theoretical L (Throughput * W):   96.5006 cars
  Difference:                       1.3782
  >> WARNING: Discrepancy detected (normal if system hasn't reached steady state).

[Energy Conservation Check]
  Total Energy Demand (Served EVs): 235477.18 kWh
  Total Energy Sold (Piles):        239770.83 kWh
  >> FAIL: Energy conservation violated.
